In [1]:
platformID = 'FBE'


In [2]:
from datetime import datetime
import pandas as pd

import os 

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from functions import lookup_loader, execute_sql_query, compare_or_update_reference
import test_functions

from config import gam_info

In [4]:
lookup = lookup_loader(gam_info, platformID, 
                       with_country=True, country_col='YT-_FBE_codes')
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test FBE_1c_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_2 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1c_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_5 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1c_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_8 passed: No missing values in lookup.
...updating logbook...



## country

In [5]:
sql_query = f"""
    SELECT 
        week_commencing,
        page_id ,
        page_name,
        page_fans_country_total AS page_fans_country,
        country_code AS fb_metric_breakdown
    FROM
        redshiftdb.central_insights.adverity_social_facebook_page_fans_by_country
    WHERE
        week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+facebook_country_raw['page_id']
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

facebook_country_raw = pd.read_csv(file, keep_default_na=False, dtype={"page_id": "string"}).drop_duplicates()
facebook_country_raw['week_commencing'] = pd.to_datetime(facebook_country_raw['week_commencing'])
facebook_country_raw = facebook_country_raw.rename(columns={'page_id': 'Channel ID',
                                                            'page_name': 'Channel Name',
                                                            'week_commencing': 'w/c',
                                                            'fb_metric_breakdown': 'YT-_FBE_codes'})



no redshift connection using last time queried


In [6]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_country_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_9",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               main_data=facebook_country_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_country_raw, 
                           numeric_columns=['page_fans_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_country_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test FBE_1c_9 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
          Start        End          Channel ID        w/c
1573 2025-04-02 2025-08-19  FBE171824429536304 2025-07-21
1576 2025-04-02 2025-08-19  FBE171824429536304 2025-08-11
1577 2025-04-02 2025-08-19  FBE171824429536304 2025-08-18

Summary of missing weeks per Channel ID:
           Channel ID  missing_week_count
0  FBE171824429536304                   3
...updating logbook...

✅ Test FBE_1c_11 passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test FBE_1c_12 passed: No combinations occurs more than once a week.
...updating logbook...



In [7]:
# filter to relevant pageID's
facebook_country = facebook_country_raw[facebook_country_raw['Channel ID'].isin(channel_ids)]
test_functions.test_inner_join(facebook_country, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='calculating country %')

facebook_country = facebook_country.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])



Inner join test FBE_1c_13 failed: Issues found.
Issues with df_left (rows present in df_left but not in df_right)
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...



In [8]:

# Group by specified columns and sum the fb_metric_value
facebook_country_sum = facebook_country.groupby(['Channel ID', 'w/c'])\
                                        .agg(Sum_page_fans_country=('page_fans_country', 'sum'))\
                                        .reset_index()
facebook_country = facebook_country.merge(facebook_country_sum, 
                                          how='inner', on= ['Channel ID', 'w/c'])
test_functions.test_inner_join(facebook_country, facebook_country_sum, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_14", 
                               test_step='calculating country %')

facebook_country['country_%'] = facebook_country['page_fans_country']/facebook_country['Sum_page_fans_country']

### RUN TESTS
test_functions.test_percentage(facebook_country, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15", 
                               test_step='summing country % per week & account')

✅ Inner join test FBE_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



In [9]:
facebook_country

,w/c,Channel ID,Channel Name,page_fans_country,PlaceID,Sum_page_fans_country,country_%
0,2025-03-31,FBE237388053065908,BBC Culture,16122,MAL,1234268,0.013062
1,2025-03-31,FBE146214266618,BBC Strictly Come Dancing,4859,NIG,1219009,0.003986
2,2025-03-31,FBE1648071085436082,BBC Rip Off Britain,22,TUR,55848,0.000394
3,2025-03-31,FBE124158667615790,BBC News Ukrainian,11772,ITA,573511,0.020526
4,2025-03-31,FBE147105585468223,BBC Worklife,2904,TUN,332243,0.008741
...,...,...,...,...,...,...,...
148981,2025-12-01,FBE160817274042538,BBC News Gahuza,1137,MAD,873352,0.001302
148982,2025-12-01,FBE236659822607,BBC Hausa,5793,INO,8318077,0.000696
148983,2025-12-01,FBE114577681901041,Later with Jools Holland,250,UAE,194491,0.001285
148984,2025-12-01,FBE114577681901041,Later with Jools Holland,581,FIN,194491,0.002987


In [10]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
facebook_country[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv",
                        index=None)

In [11]:
facebook_country.sort_values('w/c')['w/c'].unique()

<DatetimeArray>
['2025-03-31 00:00:00', '2025-04-07 00:00:00', '2025-04-14 00:00:00',
 '2025-04-21 00:00:00', '2025-04-28 00:00:00', '2025-05-05 00:00:00',
 '2025-05-12 00:00:00', '2025-05-19 00:00:00', '2025-05-26 00:00:00',
 '2025-06-02 00:00:00', '2025-06-09 00:00:00', '2025-06-16 00:00:00',
 '2025-06-23 00:00:00', '2025-06-30 00:00:00', '2025-07-07 00:00:00',
 '2025-07-14 00:00:00', '2025-07-21 00:00:00', '2025-07-28 00:00:00',
 '2025-08-04 00:00:00', '2025-08-11 00:00:00', '2025-08-18 00:00:00',
 '2025-08-25 00:00:00', '2025-09-01 00:00:00', '2025-09-08 00:00:00',
 '2025-09-15 00:00:00', '2025-09-22 00:00:00', '2025-09-29 00:00:00',
 '2025-10-06 00:00:00', '2025-10-13 00:00:00', '2025-10-20 00:00:00',
 '2025-10-27 00:00:00', '2025-11-03 00:00:00', '2025-11-10 00:00:00',
 '2025-11-17 00:00:00', '2025-11-24 00:00:00', '2025-12-01 00:00:00',
 '2025-12-08 00:00:00']
Length: 37, dtype: datetime64[ns]